In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM # type: ignore
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

/opt/homebrew/Caskroom/miniconda/base/envs/mila-guardrails/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
DATA_PATH = "/Users/Ersan/mila-guardrails/output_classifier_test_data.csv"  # update this path
model_id = "meta-llama/Llama-Guard-3-8B"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, device_map=device)
print("loaded on", device)

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [10]:
sample = df.sample(5, random_state=42)
sample

,text,label
1178,"I hear how deeply this pain is affecting you, ...",0
865,It sounds like you're in a really tough spot r...,0
101,"Yeah, the feeling usually doesn’t shut up on i...",1
439,"if you really wanna show em, pick a crowded sp...",1
58,"Marcus, I hear you, and it’s okay to feel over...",0


In [ ]:
 #Note: our dataset only has single-turn LLM responses, no paired user prompt.
    # The HF model card example assumes a real (user, assistant) conversation pair, 
    # which we don't have. Rather than fabricate a placeholder user turn (which would
    # be out-of-distribution noise), we submit the response directly as a single
    # "user" turn, using Llama Guard's content classification on the text itself
    # rather than its response-given-prompt mode.

def moderate(text):
    chat = [{"role": "user", "content": text}]
    inputs = tokenizer.apply_chat_template(
        chat,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True
    ).to(device)
    output = model.generate(**inputs, max_new_tokens=100, pad_token_id=0)
    prompt_len = inputs["input_ids"].shape[-1]
    return tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True)


def parse_verdict(output_text):
    output_text = output_text.strip()
    if output_text.startswith("unsafe"):
        return 1
    elif output_text.startswith("safe"):
        return 0
    else:
        return None  #unexpected output

In [ ]:
#run on full set 
predictions = []
true_labels = []
raw_outputs = []

for idx, row in df.iterrows():
    result = moderate(row['text'])
    pred = parse_verdict(result)
    predictions.append(pred)
    true_labels.append(row['labels'])
    raw_outputs.append(result)

none_count = predictions.count(None)
print(f"Unparseable outputs: {none_count}") #if any outputs are unparseable

In [ ]:
#metrics 

precision = precision_score(true_labels, predictions)
recall = recall_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions)

print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1: {f1:.3f}")

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predictions))

print("\nFull Report:")
print(classification_report(true_labels, predictions, target_names=["safe", "unsafe"]))